# Exercise 00
## PyTorch Introduction - Part II: Autograd and the GPU

The first notebook treated PyTorch as a fast NumPy. This notebook covers what makes it a deep learning framework: automatic differentiation and the GPU.

### Goals of this notebook

- Understand how autograd builds a graph and computes gradients.
- Know which operations are differentiable, with examples from detection, segmentation and tracking, and recognize the typical pitfalls.
- Control gradients: `no_grad`, `detach`, `eval()`, and freezing a pretrained backbone.
- Time CPU and GPU code correctly.
- Know the trade-offs between autograd and hand-derived gradients.

This notebook has no tasks. Run the cells, change them and play with them.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF

USING_COLAB = "google.colab" in sys.modules

if USING_COLAB:
    import os
    from google.colab import drive

    gdrive_path = "/content/gdrive/MyDrive/cv3dst/exercise_00"

    # This will mount your google drive under 'MyDrive'
    drive.mount("/content/gdrive", force_remount=True)
    # Navigate to the exercise folder and check manually that all files are present
    os.chdir(gdrive_path)
    print(sorted(os.listdir()))

root_dir = Path.cwd().parent  # the cv3dst folder
dataset_dir = root_dir / "datasets"

torch.set_printoptions(precision=3, sci_mode=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}, device: {device}")

## 1. The computational graph

We create two tensors `a` and `b` with `requires_grad=True`. This tells `autograd` to record every operation on them in a *computational graph*. We then compute

$Q = 3a^3 - b^2$

and let `autograd` compute the gradients of $Q$ with respect to `a` and `b`:

$\frac{\partial Q}{\partial a} = 9a^2 \qquad \frac{\partial Q}{\partial b} = -2b$

In [ ]:
a = torch.tensor([2.0, 3.0], requires_grad=True)
b = torch.tensor([6.0, 4.0], requires_grad=True)

Q = 3 * a**3 - b**2
print(f"Q = {Q}")
print(f"Q.grad_fn = {Q.grad_fn}")  # the last operation that created Q
print(f"a is a leaf: {a.is_leaf}, Q is a leaf: {Q.is_leaf}")

# backward() needs a scalar, so we sum over all elements of Q
Q.sum().backward()
print(f"dQ/da = {a.grad}, expected {9 * a.detach()**2}")
print(f"dQ/db = {b.grad}, expected {-2 * b.detach()}")

Tensors that you create yourself are *leaves* of the graph. `backward()` stores the gradients in the `.grad` attribute of the leaves only; intermediate results such as `Q` do not keep their gradients.

Gradients are **accumulated**: every call to `backward()` adds to `.grad`. If you run the next cell twice, the gradients grow. This is why the training loop resets the gradients in every step with `optimizer.zero_grad()`.

In [ ]:
Q = 3 * a**3 - b**2
Q.sum().backward()
print(f"dQ/da after a second backward pass: {a.grad}")

a.grad = None  # reset the gradients
b.grad = None

Tensors with `requires_grad=False` (the default) do not get a gradient.

In [ ]:
a = torch.tensor([2.0, 3.0], requires_grad=True)
b = torch.tensor([6.0, 4.0])

Q = 3 * a**3 - b**2
Q.sum().backward()
print(f"dQ/da = {a.grad}, dQ/db = {b.grad}")

### Checking gradients numerically

How do we know that the gradients are correct? The derivative can be approximated with *finite differences*: $\frac{\partial f}{\partial x_i} \approx \frac{f(x + \epsilon e_i) - f(x - \epsilon e_i)}{2 \epsilon}$, where $e_i$ is the $i$-th unit vector. Autograd computes the exact derivative, so both should agree. [`torch.autograd.gradcheck`](https://pytorch.org/docs/stable/generated/torch.autograd.gradcheck.gradcheck.html) does this comparison for you; it is the standard way to test a custom differentiable function. It needs `float64`, because finite differences are too inaccurate in `float32`.

In [ ]:
def f(x):
    return (x.sin() * x.exp()).sum()


x = torch.randn(5, dtype=torch.float64, requires_grad=True)
f(x).backward()

eps = 1e-6
with torch.no_grad():
    numeric = torch.stack([(f(x + eps * e) - f(x - eps * e)) / (2 * eps) for e in torch.eye(5, dtype=torch.float64)])
print(f"autograd:           {x.grad}")
print(f"finite differences: {numeric}")
print(f"gradcheck passed: {torch.autograd.gradcheck(f, (x,))}")

## 2. Differentiable image operations: interpolation

Images are only defined at integer pixel positions, but detection and tracking constantly need values in between: at the corners of a predicted box, or in a frame that is shifted by a fraction of a pixel. **Bilinear interpolation** computes such a value as a weighted average of the four neighboring pixels. The weights depend smoothly on the position, so the result is differentiable with respect to both the pixel values *and* the position.

[`F.grid_sample`](https://pytorch.org/docs/stable/generated/torch.nn.functional.grid_sample.html) samples an image `(B, C, H, W)` at the positions given in a `grid` of shape `(B, H_out, W_out, 2)`. Two details are important:
- The positions are `(x, y)` pairs in **normalized coordinates**: $-1$ is the left (top) border and $+1$ the right (bottom) border of the image, independent of its size.
- With `align_corners=False` (the default and our choice), $\pm 1$ refers to the outer *edges* of the border pixels. With `align_corners=True`, it refers to their *centers*. Mixing the two conventions shifts your results by half a pixel.

[`F.affine_grid`](https://pytorch.org/docs/stable/generated/torch.nn.functional.affine_grid.html) creates such a grid from an affine transformation $\theta$ of shape `(B, 2, 3)`. The identity $\theta = [I \mid 0]$ samples the image unchanged, the last column shifts it.

We create a synthetic frame with a few blobs and a second frame in which the camera moved by a few pixels.

In [ ]:
H, W = 96, 96


def make_frame(dx: float, dy: float, sigma: float = 3.0) -> torch.Tensor:
    # eight Gaussian blobs, shifted by (dx, dy) pixels
    centers = torch.rand(8, 2, generator=torch.Generator().manual_seed(0)) * 60 + 18
    ys, xs = torch.arange(H)[:, None], torch.arange(W)[None, :]
    dist2 = (xs - centers[:, 0, None, None] - dx) ** 2 + (ys - centers[:, 1, None, None] - dy) ** 2
    return torch.exp(-dist2 / (2 * sigma**2)).sum(dim=0)[None, None]  # (1, 1, H, W)


def warp(frame: torch.Tensor, t: torch.Tensor, mode: str = "bilinear") -> torch.Tensor:
    # shift the frame by t, given in normalized coordinates
    theta = torch.cat([torch.eye(2), t[:, None]], dim=1)[None]  # (1, 2, 3)
    grid = F.affine_grid(theta, frame.shape, align_corners=False)
    return F.grid_sample(frame, grid, mode=mode, align_corners=False)


frame0, frame1 = make_frame(0, 0), make_frame(5, -3)
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
for ax, img, title in zip(axs, (frame0, frame1), ("frame t", "frame t+1 (camera moved)")):
    ax.imshow(img[0, 0], cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.show()

### Aligning two frames with gradient descent

Many trackers compensate the camera motion before they associate objects between frames. We estimate the shift $t$ that aligns frame $t+1$ with frame $t$ by minimizing the squared difference of the warped frame and the reference frame with gradient descent. `grid_sample` makes the loss differentiable with respect to $t$.

In [ ]:
def align(reference, moving, mode="bilinear", t_init=None, steps=300):
    t = torch.zeros(2) if t_init is None else t_init.clone()
    t.requires_grad_(True)
    optimizer = torch.optim.Adam([t], lr=0.01)
    losses = []
    for _ in range(steps):
        loss = ((warp(moving, t, mode) - reference) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return t.detach(), losses


def to_pixels(t):
    return t * torch.tensor([W, H]) / 2  # normalized coordinates -> pixels


t, losses = align(frame0, frame1)
print(f"estimated shift: {to_pixels(t)} pixels, true shift: (5, -3)")

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
axs[0].plot(losses)
axs[0].set(title="alignment loss", xlabel="iteration", yscale="log")
axs[1].imshow((frame1 - frame0)[0, 0], cmap="coolwarm")
axs[1].set_title("difference before alignment")
axs[2].imshow((warp(frame1, t) - frame0)[0, 0], cmap="coolwarm")
axs[2].set_title("difference after alignment")
for ax in axs[1:]:
    ax.axis("off")
plt.show()

With **nearest neighbor** sampling, the sampled value jumps from one pixel to the next. Within a pixel the result does not change at all, so its derivative with respect to the position is zero, and gradient descent does not move.

In [ ]:
t, losses = align(frame0, frame1, mode="nearest")
print(f"nearest neighbor: estimated shift {to_pixels(t)} pixels, loss {losses[0]:.4f} -> {losses[-1]:.4f}")

Bilinear interpolation only looks at the four neighboring pixels, so the gradient only "sees" a small neighborhood. If the shift is larger than the structures in the image, the blobs no longer overlap and the optimization gets stuck. Blurring both frames enlarges the structures and with it the *capture range*. A coarse-to-fine scheme first aligns blurred frames and then refines the result on the sharp frames.

In [ ]:
frame2 = make_frame(20, 15)
t, _ = align(frame0, frame2)
print(f"sharp frames:         estimated shift {to_pixels(t)}, true shift (20, 15)")

blur0, blur2 = TF.gaussian_blur(frame0, 41, 10.0), TF.gaussian_blur(frame2, 41, 10.0)
t, _ = align(blur0, blur2)
print(f"blurred frames:       estimated shift {to_pixels(t)}")
t, _ = align(frame0, frame2, t_init=t)
print(f"refined on the sharp: estimated shift {to_pixels(t)}")

### Segmentation: upsampling logits

Segmentation networks often predict at a lower resolution than the image. Bilinear upsampling of the logits with [`F.interpolate`](https://pytorch.org/docs/stable/generated/torch.nn.functional.interpolate.html) is differentiable, so the loss can be computed against the full-resolution mask, and the gradient flows back to the low-resolution prediction.

In [ ]:
logits = torch.randn(1, 3, 16, 16, requires_grad=True)  # 3 classes, predicted at 1/4 resolution
mask = torch.randint(0, 3, (1, 64, 64))  # full-resolution ground truth

upsampled = F.interpolate(logits, size=(64, 64), mode="bilinear", align_corners=False)
loss = F.cross_entropy(upsampled, mask)
loss.backward()
print(f"upsampled logits {tuple(upsampled.shape)}, gradient of the logits {tuple(logits.grad.shape)}")

### Practical tip: cropping detections in one batch

Trackers compute appearance (ReID) features from crops of every detection, resized to a fixed size. The straightforward way loops over the boxes, rounds them to integer pixels and resizes every crop on its own. With `affine_grid` and `grid_sample`, all crops are sampled in one call: every box becomes an affine transformation that maps the output grid onto the box.

We use the first frame of the sequence MOT16-02 from the [MOT16](https://arxiv.org/abs/1603.00831) benchmark, which you will work with in the next exercises. The annotations of a sequence are stored in `gt/gt.txt`, one line per box with the columns: frame, track id, left, top, width, height (in pixels, starting at 1), a flag whether the box is used for evaluation, the class (1 = pedestrian) and the visibility.

For a box from $x_1$ to $x_2$ in an image of width $W$, the normalized coordinates of the box edges are $2x_1/W - 1$ and $2x_2/W - 1$. The transformation maps the output range $[-1, 1]$ onto them with the scale $(x_2 - x_1)/W$ and the translation $(x_1 + x_2)/W - 1$; the same holds for $y$.

Things to keep in mind:
- **Pixel convention**: here a box from $x_1 = 0$ to $x_2 = W$ covers the whole image, i.e. pixel $i$ covers the interval $[i, i + 1]$. Together with `align_corners=False`, this avoids half-pixel offsets.
- **Boxes partly outside the image**: `padding_mode="zeros"` (default) fills them with zeros, `"border"` repeats the border pixels.
- **Aspect ratio**: every box is resized to the same output size and gets distorted if its aspect ratio differs. ReID models therefore use person-shaped crops such as $256 \times 128$.

As always, check whether the function already exists: [`torchvision.ops.roi_align`](https://pytorch.org/vision/stable/generated/torchvision.ops.roi_align.html) with `aligned=True` uses the same pixel convention. With `sampling_ratio=1`, it samples one point per output pixel and gives the same result. By default, it averages several sampling points per output pixel, which is better when the crops are downscaled a lot.

In [ ]:
from PIL import Image
from torchvision.ops import roi_align
from torchvision.utils import draw_bounding_boxes

seq_dir = dataset_dir / "MOT16" / "train" / "MOT16-02"
frame = TF.pil_to_tensor(Image.open(seq_dir / "img1" / "000001.jpg")).float() / 255  # (3, H, W), uint8 -> float

rows = (seq_dir / "gt" / "gt.txt").read_text().splitlines()
gt = torch.tensor([[float(v) for v in row.split(",")] for row in rows])
gt = gt[(gt[:, 0] == 1) & (gt[:, 6] == 1) & (gt[:, 7] == 1) & (gt[:, 8] > 0.5)]  # visible pedestrians in frame 1
x1, y1 = gt[:, 2] - 1, gt[:, 3] - 1  # MOT coordinates start at 1
boxes = torch.stack([x1, y1, x1 + gt[:, 4], y1 + gt[:, 5]], dim=1)  # (N, 4) as (x1, y1, x2, y2)
print(f"frame {tuple(frame.shape)}, {len(boxes)} pedestrians")

plt.figure(figsize=(14, 8))
plt.imshow(draw_bounding_boxes((255 * frame).byte(), boxes, colors="yellow", width=3).permute(1, 2, 0))
plt.axis("off")
plt.show()

In [ ]:
def crop_boxes(frame: torch.Tensor, boxes: torch.Tensor, size: tuple[int, int]) -> torch.Tensor:
    # frame (C, H, W), boxes (N, 4) as (x1, y1, x2, y2) in pixels -> crops (N, C, *size)
    C, H, W = frame.shape
    x1, y1, x2, y2 = boxes.unbind(dim=1)
    theta = torch.zeros(len(boxes), 2, 3, device=frame.device)
    theta[:, 0, 0] = (x2 - x1) / W
    theta[:, 0, 2] = (x1 + x2) / W - 1
    theta[:, 1, 1] = (y2 - y1) / H
    theta[:, 1, 2] = (y1 + y2) / H - 1
    grid = F.affine_grid(theta, (len(boxes), C, *size), align_corners=False)
    return F.grid_sample(frame.expand(len(boxes), -1, -1, -1), grid, align_corners=False)  # expand: no copy


boxes.requires_grad_(True)
crops = crop_boxes(frame, boxes, (256, 128))
reference = roi_align(frame[None], [boxes.detach()], output_size=(256, 128), sampling_ratio=1, aligned=True)
print(f"crops {tuple(crops.shape)}, equal to roi_align: {torch.allclose(crops, reference, atol=1e-4)}")

# in contrast to roi_align, the crops are differentiable with respect to the boxes
crops.sum().backward()
print(f"gradient of the first three boxes:\n{boxes.grad[:3]}")

fig, axs = plt.subplots(1, len(crops), figsize=(16, 3.5))
for ax, crop in zip(axs, crops.detach()):
    ax.imshow(crop.permute(1, 2, 0))
    ax.axis("off")
plt.show()

Parts of a box outside the image would be filled with zeros; in tracking, this happens for every person who enters or leaves the image. We compare the speed with the loop version in section 7.

## 3. Non-differentiable operations and pitfalls

Training only works if the loss is differentiable with respect to the parameters. Many common operations are not, or only partially:
- Operations with **integer or boolean results**, such as `argmax`, comparisons and casts to integers, are not part of the graph.
- Operations that are **piecewise constant**, such as `round` and `floor`, have a gradient of zero almost everywhere.
- **Indexing** passes the gradient to the selected values, but not to the indices.

In [ ]:
x = torch.tensor([0.2, 1.7, -0.4], requires_grad=True)
print(f"x.argmax() = {x.argmax()}, requires_grad: {x.argmax().requires_grad}")
print(f"x > 0 = {x > 0}, requires_grad: {(x > 0).requires_grad}")

x.round().sum().backward()
print(f"gradient of round: {x.grad}")

In [ ]:
img = torch.rand(5, 5, requires_grad=True)
pos = torch.tensor([1.6, 2.3], requires_grad=True)  # a sub-pixel position (x, y), e.g. a predicted keypoint

value = img[pos[1].long(), pos[0].long()]  # integer indexing: the pixel at (x=1, y=2)
value.backward()
print(f"gradient of the image:\n{img.grad}")
print(f"gradient of the position: {pos.grad}")  # None: .long() cuts the graph

### Silent graph breaks

Some operations cut the graph without an error. The gradient then simply does not reach your parameters:
- `.item()`, `.tolist()` and `.numpy()` return python or NumPy values that are not part of the graph.
- `torch.tensor([a, b])` copies the *values* of tensors into a new leaf tensor. Use `torch.stack` or `torch.cat` instead.
- `.detach()` cuts the graph on purpose, e.g. for targets that should be treated as constants.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

print(f"torch.tensor: requires_grad = {torch.tensor([a * b, a + b]).requires_grad}")  # also prints a warning
print(f"torch.stack:  requires_grad = {torch.stack([a * b, a + b]).requires_grad}")
print(f".detach():    requires_grad = {(a * b).detach().requires_grad}")

### Non-differentiable points and NaN gradients

Functions such as `relu`, `abs` and `max` are not differentiable at single points, e.g. `abs` at 0, where the slope jumps from -1 to 1. PyTorch uses a fixed convention there (a *subgradient*), e.g. the derivative of `abs` at 0 is 0.

More dangerous are functions with an infinite derivative, such as `sqrt` at 0. A classic bug: `torch.where` selects the result of one branch, but the backward pass differentiates **both** branches and multiplies the unused one by 0. Since $0 \cdot \infty$ and $0 \cdot \text{NaN}$ are NaN, the gradient becomes NaN, even though the forward result looks fine.

In [ ]:
x = torch.tensor([-1.0, 0.0, 4.0], requires_grad=True)
y = torch.where(x > 0, x.sqrt(), 0.0)
y.sum().backward()
print(f"y = {y}, gradient {x.grad}")  # NaN, although y has no NaN

x.grad = None
y = torch.where(x > 0, x.clamp(min=1e-12).sqrt(), 0.0)  # make both branches safe
y.sum().backward()
print(f"y = {y}, gradient {x.grad}")

The same happens for Euclidean distances, e.g. between the appearance features of the same person in metric learning. If two feature vectors are identical, the derivative of the square root is infinite. `torch.linalg.vector_norm` handles this case, the manual formula does not. Clamping the squared distance before the square root avoids the problem.

In [ ]:
f1 = torch.tensor([1.0, 2.0], requires_grad=True)
f2 = torch.tensor([1.0, 2.0])  # identical features

for name, fn in [
    ("vector_norm  ", lambda d: torch.linalg.vector_norm(d)),
    ("sqrt(sum)    ", lambda d: (d**2).sum().sqrt()),
    ("clamped sqrt ", lambda d: (d**2).sum().clamp(min=1e-12).sqrt()),
]:
    f1.grad = None
    fn(f1 - f2).backward()
    print(f"{name}: gradient {f1.grad}")

### In-place operations

Many operations save their inputs or outputs for the backward pass, e.g. `exp` saves its output. Modifying such a tensor in-place raises an error in the backward pass. Use the out-of-place version (`y = y + 1` instead of `y += 1`) on tensors that are part of the graph.

In [ ]:
x = torch.randn(3, requires_grad=True)
y = x.exp()
y += 1  # in-place modification of a saved tensor
try:
    y.sum().backward()
except RuntimeError as e:
    print(f"RuntimeError: {e}")

### Soft relaxations

If a non-differentiable operation is needed in the loss, it can often be replaced by a smooth version:
- **soft-argmax** instead of `argmax`: keypoint and center-based detectors predict a heatmap with a peak at every object. The softmax over the heatmap gives a probability for every pixel, and the expected position $\sum_i p_i x_i$ is differentiable. A temperature $\tau$ controls how sharp the softmax is.
- `torch.sigmoid((x - t) / τ)` instead of the hard threshold `x > t`, e.g. for a soft segmentation mask.
- The *straight-through estimator* uses the hard operation in the forward pass and pretends it was the identity in the backward pass.

In [ ]:
ys, xs = torch.arange(H)[:, None].float(), torch.arange(W)[None, :].float()
heatmap = torch.exp(-((xs - 30.4) ** 2 + (ys - 52.7) ** 2) / (2 * 4.0**2))  # one object centered at (30.4, 52.7)
heatmap.requires_grad_(True)

idx = heatmap.argmax()  # flattened index, no gradient
print(f"argmax:      ({idx % W}, {idx // W}), requires_grad: {idx.requires_grad}")

tau = 0.05
p = torch.softmax(heatmap.flatten() / tau, dim=0).view(H, W)
x, y = (p * xs).sum(), (p * ys).sum()
print(f"soft-argmax: ({x.item():.2f}, {y.item():.2f}), requires_grad: {x.requires_grad}")

## 4. Controlling gradients

During evaluation we do not need gradients. Wrapping the code in `with torch.no_grad():` disables the tracking, which saves memory and time. `torch.inference_mode()` is a stricter and slightly faster version for pure inference. `.detach()` returns a tensor without history, and `requires_grad_(False)` stops tracking for a tensor, e.g. for parameters that should not be trained (section 5).

In [ ]:
a = torch.tensor([2.0, 3.0], requires_grad=True)

with torch.no_grad():
    Q = 3 * a**3
print(f"no_grad:        Q.requires_grad = {Q.requires_grad}")

with torch.inference_mode():
    Q = 3 * a**3
print(f"inference_mode: Q.requires_grad = {Q.requires_grad}")

### `model.eval()` is not `no_grad()`

These two are often confused, but they do different things:

| | changes | typical use |
|---|---|---|
| `model.train()` / `model.eval()` | the **behavior of layers**: dropout is switched off and batch normalization uses its running statistics in eval mode | always switch to `eval()` for validation and testing, and back to `train()` for training |
| `torch.no_grad()` | whether **gradients are tracked** | evaluation, and any computation that should not be part of the graph |

For evaluation you need both.

In [ ]:
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(4, 4), nn.Dropout(p=0.5))
x = torch.ones(1, 4)

with torch.no_grad():
    print(f"train mode + no_grad: {net(x)}")  # dropout still sets random elements to zero

net.eval()
out = net(x)
print(f"eval mode:            {out}, requires_grad: {out.requires_grad}")  # deterministic, but still tracked

## 5. Transfer learning: freezing a pretrained backbone

Training a large network from scratch needs a lot of data. Instead, we start from a *backbone* that was pretrained on a large dataset such as ImageNet and adapt it to our task. You will see both variants in this course: exercise 2 fine-tunes a pretrained ResNet for person re-identification, and exercise 4 trains a small classifier on top of frozen DINO features.

To **freeze** the backbone, we switch off the gradients of its parameters and replace the classification head with a new layer for our task. New layers have `requires_grad=True` by default. Only the trainable parameters are passed to the optimizer.

In [ ]:
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
model.requires_grad_(False)  # freeze all parameters
model.fc = nn.Linear(model.fc.in_features, 5)  # new head for 5 classes, trainable
# we stay on the CPU in this section, so that section 7 can show the warm-up of the GPU

trainable = [p for p in model.parameters() if p.requires_grad]
print(f"trainable parameters: {sum(p.numel() for p in trainable):,} of {sum(p.numel() for p in model.parameters()):,}")
optimizer = torch.optim.SGD(trainable, lr=0.01, momentum=0.9)

**Caution with batch normalization:** `requires_grad_(False)` freezes the learnable parameters, but batch normalization layers also have *running statistics* (mean and variance) that are updated in every forward pass in training mode. They are not parameters, so they are not affected by `requires_grad`. Training the head in `train()` mode silently changes the frozen backbone:

In [ ]:
x = torch.randn(8, 3, 224, 224)
y = torch.randint(0, 5, (8,))
before = model.bn1.running_mean.clone()

model.train()
loss = F.cross_entropy(model(x), y)
optimizer.zero_grad()
loss.backward()
optimizer.step()
print(f"running mean unchanged in train mode: {torch.equal(before, model.bn1.running_mean)}")

The fix: after every call of `model.train()`, switch the frozen layers back to evaluation mode.

In [ ]:
def train_with_frozen_backbone(model: nn.Module) -> None:
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()


before = model.bn1.running_mean.clone()
train_with_frozen_backbone(model)
loss = F.cross_entropy(model(x), y)
optimizer.zero_grad()
loss.backward()
optimizer.step()
print(f"running mean unchanged: {torch.equal(before, model.bn1.running_mean)}")

### Fine-tuning with parameter groups

Instead of freezing the backbone completely, you can also *fine-tune* it with a smaller learning rate than the new head. The optimizer accepts a list of *parameter groups*, each with its own settings.

In [ ]:
model.requires_grad_(True)
backbone_params = [p for name, p in model.named_parameters() if not name.startswith("fc.")]
optimizer = torch.optim.SGD(
    [
        {"params": model.fc.parameters(), "lr": 1e-2},
        {"params": backbone_params, "lr": 1e-4},
    ],
    momentum=0.9,
)
print([group["lr"] for group in optimizer.param_groups])

### Extracting intermediate features

Detectors and segmentation networks do not use the classification output of a backbone, but its feature maps at several resolutions, e.g. as input to a feature pyramid network (FPN). [`create_feature_extractor`](https://pytorch.org/vision/stable/feature_extraction.html) returns the outputs of the layers you name.

In [ ]:
from torchvision.models.feature_extraction import create_feature_extractor

model.eval()
extractor = create_feature_extractor(model, return_nodes={"layer2": "stride 8", "layer3": "stride 16", "layer4": "stride 32"})
with torch.no_grad():
    features = extractor(torch.randn(1, 3, 480, 640))
for name, f in features.items():
    print(f"{name:>9}: {tuple(f.shape)}")

## 6. Modules: parameters, buffers and the state dict

An `nn.Module` holds two kinds of tensors:
- **parameters** (`nn.Parameter`) are trained by the optimizer,
- **buffers** are part of the model's state, but are not trained, e.g. the running statistics of batch normalization. You register your own with `self.register_buffer(name, tensor)`.

Both are moved by `.to(device)` and saved in the *state dict*. The state dict maps names to tensors and is the recommended way to save a model: `torch.save(model.state_dict(), path)` and `model.load_state_dict(torch.load(path))`. Since PyTorch 2.6, `torch.load` only loads tensors and basic python types by default (`weights_only=True`), because loading arbitrary pickled python objects can execute code.

In [ ]:
class Normalize(nn.Module):
    def __init__(self, mean: list[float], std: list[float]):
        super().__init__()
        self.register_buffer("mean", torch.tensor(mean)[:, None, None])
        self.register_buffer("std", torch.tensor(std)[:, None, None])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return (x - self.mean) / self.std


net = nn.Sequential(Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]), nn.Conv2d(3, 8, 3), nn.BatchNorm2d(8))
print(f"parameters: {[name for name, _ in net.named_parameters()]}")
print(f"buffers:    {[name for name, _ in net.named_buffers()]}")
print(f"state dict: {list(net.state_dict().keys())}")

## 7. Timing CPU and GPU code

GPU operations run **asynchronously**: the python call only *launches* the computation and returns immediately, while the GPU works through its queue in the background. Python waits only when it needs a result, e.g. for `.item()`, `.cpu()` or `print`. Measuring GPU code correctly therefore needs care:

1. **Synchronize** with `torch.cuda.synchronize()` before stopping the clock, otherwise you only measure the launch.
2. **Warm up** before measuring. The first calls are slow: PyTorch initializes the CUDA context and libraries such as cuBLAS, selects kernels and grows its memory cache. With `torch.backends.cudnn.benchmark = True`, cuDNN additionally tries several convolution algorithms in the first call for every new input shape.
3. **Repeat** the measurement and take the median.

The cells below need a GPU. On Colab, select *Runtime > Change runtime type > T4 GPU*.

In [ ]:
def time_ms(fn, sync: bool) -> float:
    if sync:
        torch.cuda.synchronize()
    start = time.perf_counter()
    fn()
    if sync:
        torch.cuda.synchronize()
    return (time.perf_counter() - start) * 1e3


if device.type == "cuda":
    A = torch.randn(4096, 4096, device=device)
    B = torch.randn(4096, 4096, device=device)
    print(f"1st call, synchronized:  {time_ms(lambda: A @ B, sync=True):8.2f} ms  (includes the warm-up)")
    print(f"2nd call, synchronized:  {time_ms(lambda: A @ B, sync=True):8.2f} ms")
    print(f"3rd call, no synchronize: {time_ms(lambda: A @ B, sync=False):7.2f} ms  (only the launch!)")
else:
    print("No GPU available. On Colab, select 'Runtime > Change runtime type > T4 GPU'.")

**CUDA events** measure the time on the GPU itself. They are recorded into the GPU queue like any other operation.

In [ ]:
if device.type == "cuda":
    start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(10):
        A @ B
    end.record()
    torch.cuda.synchronize()  # wait until the end event is reached
    print(f"matrix product: {start.elapsed_time(end) / 10:.2f} ms")

In practice, use [`torch.utils.benchmark`](https://pytorch.org/docs/stable/benchmark_utils.html). Its `Timer` synchronizes the GPU, runs the statement a few times before it measures, and repeats the measurement until the result is reliable. We use it for the rest of the notebook.

### When is the GPU faster?

Every GPU operation has a fixed launch overhead of a few microseconds. For small inputs, this overhead dominates and the CPU is faster. For large inputs, the thousands of GPU cores win by orders of magnitude.

In [ ]:
from torch.utils import benchmark


def measure_ms(stmt: str, **variables) -> float:
    return benchmark.Timer(stmt=stmt, globals=variables).blocked_autorange(min_run_time=0.2).median * 1e3


sizes = [16, 32, 64, 128, 256, 512, 1024, 2048]
cpu_ms = [measure_ms("A @ A", A=torch.randn(n, n)) for n in sizes]
plt.figure(figsize=(7, 4))
plt.loglog(sizes, cpu_ms, marker="o", label="CPU")
if device.type == "cuda":
    gpu_ms = [measure_ms("A @ A", A=torch.randn(n, n, device=device)) for n in sizes]
    plt.loglog(sizes, gpu_ms, marker="o", label="GPU")
plt.xlabel("matrix size n (n x n)")
plt.ylabel("time [ms]")
plt.legend()
plt.show()

### Moving data costs time

Copying data between CPU and GPU memory is slow compared to computing on the GPU. For a single cheap operation, the copy takes much longer than the operation itself. Therefore: move the data to the GPU once, keep it there, and move whole batches instead of single samples. *Pinned* (page-locked) CPU memory speeds up the copy; the `DataLoader` uses it with `pin_memory=True` (next notebook).

In [ ]:
if device.type == "cuda":
    x_cpu = torch.randn(4096, 4096)
    x_gpu = x_cpu.to(device)
    print(f"x * 2 on the CPU:       {measure_ms('x * 2', x=x_cpu):6.2f} ms")
    print(f"x * 2 on the GPU:       {measure_ms('x * 2', x=x_gpu):6.2f} ms")
    print(f"copy to the GPU:        {measure_ms('x.to(device)', x=x_cpu, device=device):6.2f} ms")
    print(f"copy from pinned memory: {measure_ms('x.to(device)', x=x_cpu.pin_memory(), device=device):5.2f} ms")

### Example: cropping detections

Let us come back to the crops of the MOT16 frame from section 2 and compare a loop over the boxes, our batched version and `roi_align` in two scenarios:
- a tracker crops the annotated pedestrians to the ReID size of $256 \times 128$ pixels,
- a two-stage detector crops 500 box proposals around the pedestrians to $7 \times 7$ (on a feature map in practice, on the image here).

In [ ]:
def crop_boxes_loop(frame: torch.Tensor, boxes: torch.Tensor, size: tuple[int, int]) -> torch.Tensor:
    crops = []
    for x1, y1, x2, y2 in torchvision.ops.clip_boxes_to_image(boxes, frame.shape[-2:]).int().tolist():
        crop = frame[None, :, y1:y2, x1:x2]
        crops.append(F.interpolate(crop, size, mode="bilinear", align_corners=False))
    return torch.cat(crops)


torch.manual_seed(0)
pedestrians = boxes.detach()
shift = 20 * torch.randn(500, 2)
proposals = pedestrians[torch.randint(len(pedestrians), (500,))] + torch.cat([shift, shift], dim=1)

scenarios = {
    f"{len(pedestrians)} pedestrians -> 256 x 128": (pedestrians, (256, 128)),
    "500 proposals -> 7 x 7": (proposals, (7, 7)),
}
methods = {
    "loop": crop_boxes_loop,
    "batched": crop_boxes,
    "roi_align": lambda f, b, size: roi_align(f[None], [b], size, sampling_ratio=1, aligned=True),
}

for name, (scenario_boxes, size) in scenarios.items():
    print(name)
    for dev in ["cpu", "cuda"] if device.type == "cuda" else ["cpu"]:
        f, b = frame.to(dev), scenario_boxes.to(dev)
        times = [f"{m}: {measure_ms('fn(f, b, size)', fn=fn, f=f, b=b, size=size):6.2f} ms" for m, fn in methods.items()]
        print(f"  {dev:>4}  " + "   ".join(times))

Batching pays off when there are **many small operations**: for 500 small crops, the loop spends most of its time on the overhead of python and of launching GPU kernels for every box, and the batched version is more than an order of magnitude faster. For a few large crops, every crop is already a large operation and the loop is competitive; here the advantages of the batched version are its sub-pixel accuracy and the gradient with respect to the boxes. `roi_align` is in the same range as the batched version. If you do not need gradients for the boxes, it is the better choice: it is tested, and by default it averages several sampling points per output pixel.

## 8. Autograd vs. hand-derived gradients

For a single equation, we can also derive the gradient by hand and implement the resulting *closed-form* expression directly. Autograd instead applies the chain rule to every elementary operation of the forward pass. Both are exact, but they have different trade-offs:

| | autograd | hand-derived gradient |
|---|---|---|
| **Flexibility** | works for any composition of operations; change the model and the gradient follows | must be derived again for every change of the equation, which is error-prone |
| **Scalability** | one backward pass computes the gradients of *all* parameters, at a cost of a small multiple of the forward pass | the computational cost is similar, but deriving and maintaining it by hand does not scale to networks with many layers |
| **Speed** | every elementary operation is a separate step with its own overhead | terms can be simplified and combined into fewer operations |
| **Memory** | stores the intermediate result of every operation for the backward pass | stores only what the simplified expression needs |
| **Numerics** | follows the formula as written: an unstable formula gives unstable gradients | the simplified expression is often more stable |

We compare both for the mean cross-entropy loss of $N$ samples with logits $z$ and labels $y$, written with elementary operations:

$$L = -\frac{1}{N} \sum_i \log p_{i, y_i} \qquad p_{ij} = \frac{e^{z_{ij}}}{\sum_k e^{z_{ik}}}$$

Autograd differentiates through `exp`, `sum`, the division, `log` and the one-hot selection one by one. By hand, all of this simplifies to the well-known expression

$$\frac{\partial L}{\partial z} = \frac{1}{N} \left( p - \text{onehot}(y) \right)$$

In [ ]:
def cross_entropy_manual(z, y):
    p = z.exp() / z.exp().sum(dim=1, keepdim=True)
    return -(F.one_hot(y, z.shape[1]) * p.log()).sum(dim=1).mean()


def ce_autograd(z, y):
    z = z.detach().requires_grad_(True)
    loss = cross_entropy_manual(z, y)
    loss.backward()
    return loss.detach(), z.grad


def ce_closed_form(z, y):
    onehot = F.one_hot(y, z.shape[1])
    log_p = torch.log_softmax(z, dim=1)
    loss = -(onehot * log_p).sum(dim=1).mean()
    return loss, (log_p.exp() - onehot) / len(y)


def ce_builtin(z, y):
    z = z.detach().requires_grad_(True)
    loss = F.cross_entropy(z, y)
    loss.backward()
    return loss.detach(), z.grad


z, y = torch.randn(64, 10), torch.randint(0, 10, (64,))
_, grad_autograd = ce_autograd(z, y)
_, grad_closed_form = ce_closed_form(z, y)
print(f"same gradient: {torch.allclose(grad_autograd, grad_closed_form, atol=1e-6)}")

We measure the time to compute the loss and the gradient for a small and a large problem, e.g. 4096 samples with 1000 classes. On the GPU, we also measure the peak memory. For comparison, we include PyTorch's built-in `F.cross_entropy`.

In [ ]:
def peak_memory_mb(fn, *args) -> float:
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    before = torch.cuda.memory_allocated()
    fn(*args)
    torch.cuda.synchronize()
    return (torch.cuda.max_memory_allocated() - before) / 1e6


methods = {"autograd": ce_autograd, "closed form": ce_closed_form, "F.cross_entropy": ce_builtin}
for dev in ["cpu", "cuda"] if device.type == "cuda" else ["cpu"]:
    for n, c in [(64, 10), (4096, 1000)]:
        z, y = torch.randn(n, c, device=dev), torch.randint(0, c, (n,), device=dev)
        print(f"{dev}, {n} samples, {c} classes")
        for name, fn in methods.items():
            msg = f"  {name:>15}: {measure_ms('fn(z, y)', fn=fn, z=z, y=y):7.3f} ms"
            if dev == "cuda" and n > 64:
                msg += f", peak memory {peak_memory_mb(fn, z, y):6.1f} MB"
            print(msg)

The closed form is faster and needs less memory, because it computes fewer intermediate results and does not store them for a backward pass. For the small problem, the overhead of autograd per operation dominates.

The simplified expression is also more stable. With large logits, `exp` overflows in the manual formula, and autograd faithfully differentiates the overflow:

In [ ]:
z, y = 100 * torch.randn(8, 10), torch.randint(0, 10, (8,))
print(f"autograd:    NaN in the gradient: {ce_autograd(z, y)[1].isnan().any()}")
print(f"closed form: NaN in the gradient: {ce_closed_form(z, y)[1].isnan().any()}")

### The best of both

The two approaches are not exclusive. Autograd itself is built from hand-derived gradients: every PyTorch operation has a closed-form derivative, and autograd only chains them. `F.cross_entropy` is the fastest above, because it is a single operation with a hand-derived, simplified and stable backward pass.

You can do the same for your own equations with [`torch.autograd.Function`](https://pytorch.org/docs/stable/notes/extending.html): `forward` computes the result and saves what the backward pass needs, and `backward` receives the gradient of the loss with respect to the output and returns the gradients with respect to the inputs (`None` for inputs without a gradient, such as the labels). The function then works with autograd like any other operation. `gradcheck` from section 1 verifies the hand-derived backward pass.

In [ ]:
class CrossEntropy(torch.autograd.Function):
    @staticmethod
    def forward(ctx, z, y):
        onehot = F.one_hot(y, z.shape[1])
        log_p = torch.log_softmax(z, dim=1)
        ctx.save_for_backward(log_p, onehot)
        return -(onehot * log_p).sum(dim=1).mean()

    @staticmethod
    def backward(ctx, grad_output):
        log_p, onehot = ctx.saved_tensors
        return grad_output * (log_p.exp() - onehot) / len(onehot), None


z = torch.randn(16, 5, dtype=torch.float64, requires_grad=True)
y = torch.randint(0, 5, (16,))
print(f"gradcheck passed: {torch.autograd.gradcheck(CrossEntropy.apply, (z, y))}")

loss = 2 * CrossEntropy.apply(z, y)  # works inside a larger graph
loss.backward()
print(f"same as F.cross_entropy: {torch.allclose(z.grad, torch.autograd.grad(2 * F.cross_entropy(z, y), z)[0])}")

In practice: rely on autograd for whole models. Write a custom `autograd.Function` only for a single operation that is a bottleneck in time or memory, or whose naive gradient is unstable, and always check it with `gradcheck`.

## Summary

- Autograd records the operations on tensors with `requires_grad=True` and computes the gradients of a scalar with `backward()`. Gradients accumulate; `gradcheck` verifies them numerically.
- Bilinear interpolation (`grid_sample`, `interpolate`) is differentiable with respect to the values and the sampling positions. Nearest neighbor sampling, rounding and integer indexing are not.
- Watch out for silent graph breaks (`.item()`, `torch.tensor`), NaN gradients (`sqrt` at 0 in `torch.where` or distances) and in-place operations.
- `model.eval()` changes the behavior of layers, `torch.no_grad()` switches off gradient tracking.
- Freeze a backbone with `requires_grad_(False)`, pass only the trainable parameters to the optimizer, and keep frozen batch normalization layers in `eval()` mode.
- GPU code runs asynchronously: synchronize, warm up and repeat when you measure it, e.g. with `torch.utils.benchmark`. Keep data on the GPU and process it in batches.
- Autograd is flexible and scales to whole networks; a hand-derived gradient of a single equation can be faster, leaner and more stable. `torch.autograd.Function` combines both.

Further topics that are worth a look: gradient clipping with `torch.nn.utils.clip_grad_norm_`, mixed precision training with `torch.autocast`, and `torch.compile`.

## References

1. [PyTorch Tutorial: Automatic Differentiation with torch.autograd](https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)
2. [PyTorch Tutorial: Transfer Learning](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
3. [PyTorch Benchmark Recipe](https://pytorch.org/tutorials/recipes/recipes/benchmark.html)
4. [PyTorch: Extending torch.autograd](https://pytorch.org/docs/stable/notes/extending.html)
5. Milan et al., *MOT16: A Benchmark for Multi-Object Tracking*, arXiv 2016